In [ ]:
is_causal = True # Set to False for future leakage

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_, spectral_norm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import sys
sys.path.append("../EDAF-Group_6-PV_Forecasting")

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
pv_data = pd.read_csv("../data/PV_2022_hourly.csv")
weather_data = pd.read_csv("../data/Weather_clean_NEMS.csv")

In [ ]:
df_pv = pd.DataFrame(pv_data)
df_weather = pd.DataFrame(weather_data)

In [ ]:
df_pv.info()

In [ ]:
weather_data.info()

In [ ]:
df_pv.head()

In [ ]:
df_weather.head()

In [ ]:
df_pv.shape

In [ ]:
df_weather.shape

In [ ]:
df_pv[-10:]

In [ ]:
df_pv["TimestampInUtc"] = pd.to_datetime(df_pv["TimestampInUtc"])

last_time = df_pv["TimestampInUtc"].iloc[-1]

new_rows = pd.DataFrame({
    "TimestampInUtc": [
        last_time + pd.Timedelta(hours=1),
        last_time + pd.Timedelta(hours=2)
    ],
    "pv": [0, 0]
})

df_pv = pd.concat([df_pv, new_rows], ignore_index=True)

In [ ]:
df_pv[-10:]

In [ ]:
df_pv.shape

In [ ]:
df_weather.shape

In [ ]:
df_merged = pd.concat([df_pv, df_weather], axis=1)

In [ ]:
df_merged.head()

In [ ]:
df_merged.iloc[-10:]

In [ ]:
df_merged.shape

In [ ]:
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])
df_merged["hour"] = df_merged["timestamp"].dt.hour
df_merged["day_of_year"] = df_merged["timestamp"].dt.dayofyear
df_merged["month"] = df_merged["timestamp"].dt.month

In [ ]:
df_merged.columns

In [ ]:
df_merged["hour_sin"]  = np.sin(2 * np.pi * df_merged["hour"] / 24)
df_merged["hour_cos"]  = np.cos(2 * np.pi * df_merged["hour"] / 24)

df_merged["month_sin"] = np.sin(2 * np.pi * df_merged["month"] / 12)
df_merged["month_cos"] = np.cos(2 * np.pi * df_merged["month"] / 12)

df_merged["doy_sin"] = np.sin(2 * np.pi * df_merged["day_of_year"] / 365)
df_merged["doy_cos"] = np.cos(2 * np.pi * df_merged["day_of_year"] / 365)

In [ ]:
df_merged.iloc[-744:]

In [ ]:
df_merged.drop(columns=['TimestampInUtc', 'timestamp', 'hour', 'day_of_year', 'month'], inplace=True)

In [ ]:
df_test = df_merged.iloc[-744:].copy()
df_train = df_merged.iloc[:-744].copy()

In [ ]:
df_train.columns

In [ ]:
TARGET_COL = "pv"

CONTINUOUS_FEATURES = [
    "Temperature", "Sunshine Duration", "Shortwave Radiation",
    "Direct Shortwave Radiation", "Diffuse Shortwave Radiation",
    "Snowfall Amount", "Relative Humidity", "Cloud Cover Total"
]

CYCLIC_FEATURES = [
    "hour_sin", "hour_cos",
    "month_sin", "month_cos",
    "doy_sin", "doy_cos"
]

CATEGORICAL_FEATURES = []

In [ ]:
def prepare_data(df_train, df_test):

    # ---------- continuous ----------
    Xc_train = df_train[CONTINUOUS_FEATURES].values
    Xc_test  = df_test[CONTINUOUS_FEATURES].values

    # ---------- cyclic ----------
    Xcy_train = df_train[CYCLIC_FEATURES].values
    Xcy_test  = df_test[CYCLIC_FEATURES].values

    # ---------- target ----------
    y_train = df_train[[TARGET_COL]].values
    y_test  = df_test[[TARGET_COL]].values

    y_train_unscaled = y_train.copy()

    # ---------- scalers ----------
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    # fit on train
    Xc_train_scaled = scaler_X.fit_transform(Xc_train)
    Xc_test_scaled  = scaler_X.transform(Xc_test)

    y_train_scaled = scaler_y.fit_transform(y_train)
    y_test_scaled  = scaler_y.transform(y_test)

    # ---------- combine ----------
    X_train = np.concatenate([Xc_train_scaled, Xcy_train], axis=1).astype(np.float32)
    X_test  = np.concatenate([Xc_test_scaled,  Xcy_test],  axis=1).astype(np.float32)

    y_train = y_train_scaled.astype(np.float32)
    y_test  = y_test_scaled.astype(np.float32)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
        "y_train_unscaled": y_train_unscaled
    }

In [ ]:
data = prepare_data(df_train, df_test)

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

In [ ]:
LOOKBACK = 168   # 7 days (tune: 24, 48, 72, 168)
HORIZON  = 24

In [ ]:
def create_sequences_multistep(X, y, lookback, horizon=24):
    Xs, ys = [], []
    last_i = len(X) - lookback - horizon + 1
    for i in range(last_i):
        Xs.append(X[i:i+lookback])                     # (lookback, n_features)
        ys.append(y[i+lookback:i+lookback+horizon])    # (horizon,)
    return np.array(Xs), np.array(ys)

In [ ]:
X_train_seq, y_train_seq = create_sequences_multistep(X_train, y_train, LOOKBACK, HORIZON)
X_test_seq, y_test_seq = create_sequences_multistep(X_test, y_test, LOOKBACK, HORIZON)

In [ ]:
print("Train seq:", X_train_seq.shape, y_train_seq.shape)
print("Test  seq:", X_test_seq.shape,  y_test_seq.shape)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class Chomp1d(nn.Module):
    """Remove extra padding from causal conv to keep output length same as input"""
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout, causal):
        super().__init__()
        # Replace weight_norm with spectral_norm
        self.conv1 = spectral_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        if causal:
            self.chomp1 = Chomp1d(padding)
        else:
            self.chomp1 = nn.Identity()
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = spectral_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        if causal:
            self.chomp2 = Chomp1d(padding)
        else:
            self.chomp2 = nn.Identity()
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

In [ ]:
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            dilation = 2**i
            
            if is_causal:
                padding_value = (kernel_size - 1) * dilation
            else:
                # symmetric "same" padding for non-causal
                padding_value = ((kernel_size - 1) * dilation) // 2

            layers.append(
                TemporalBlock(
                    in_ch, out_ch, kernel_size,
                    stride=1, dilation=dilation,
                    padding=padding_value,
                    dropout=dropout,
                    causal=is_causal
                )
            )

        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # TCN expects (batch, channels, seq_len)
        x = x.transpose(1, 2)  # (B, seq_len, features) → (B, features, seq_len)
        y = self.network(x)
        y = y[:, :, -1]        # take last time step
        y = self.fc(y)
        return y

In [ ]:
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        return base_lr * (epoch + 1) / warmup_epochs

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n

In [ ]:
# Hyperparams
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3

NUM_CHANNELS = [24, 24, 24, 24]
KERNEL_SIZE = 3
DROPOUT = 0.2
CAUSAL = is_causal

criterion = nn.MSELoss()

def run_timeseries_kfold(X_seq, y_seq, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_losses = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_seq)):
        train_ds = SeqDataset(X_seq[tr_idx], y_seq[tr_idx])
        val_ds   = SeqDataset(X_seq[va_idx], y_seq[va_idx])

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

        model = TCN(
            input_size=X_seq.shape[-1],
            output_size=y_seq.shape[-1],
            num_channels=NUM_CHANNELS,
            kernel_size=KERNEL_SIZE,
            dropout=DROPOUT
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val = float("inf")
        best_state = None

        for epoch in range(EPOCHS):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            va_loss = eval_one_epoch(model, val_loader, criterion)

            if va_loss < best_val:
                best_val = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        fold_losses.append(best_val)
        print(f"Fold {fold+1}/{n_splits} | best val MSE: {best_val:.5f}")

    return fold_losses

fold_losses = run_timeseries_kfold(X_train_seq, y_train_seq, n_splits=5)
print("Fold losses:", fold_losses)
print("Mean ± std:", np.mean(fold_losses), "±", np.std(fold_losses))
